# Realistic Stress-Testing: Simple and Medium Networks

This notebook runs the named disruption-scenario library from `scripts/disruption_scenarios.py` against the simple and medium realistic datasets generated in `05_realistic_operational_data`, using the exact same `build_and_solve_ttr`/`build_and_solve_tts` engine as `02_stress_testing (small network)`. Instead of disrupting one random node per run, we compare three kinds of scenarios:

- **Single-supplier failure** — the original baseline (one node, at a time)
- **Regional disruption** — every supplier in a region fails at once (only meaningful on networks with a `region` tag, like the medium network)
- **Material-wide shortage** — every supplier of one material type fails at once

A few of the regional scenarios are drawn from real historical events (see `scripts/disruption_scenarios.py`'s `_REAL_WORLD_EVENTS` for citations).

> **Modeling caveat:** the flow-balance constraint lets a disrupted node keep shipping from its pre-existing on-hand inventory even though its own production is halted, and that inventory is a *fixed* quantity — not scaled by how long the disruption lasts. A short disruption at a well-stocked node can show *zero* impact while a longer one at the same node shows real impact. That's expected behavior of the underlying LP, not a sign that a scenario "isn't working" — watch for it below.

## Cluster Configuration
This notebook was tested on the following Databricks cluster configuration:
- **Databricks Runtime Version:** 17.3 LTS ML (includes Apache Spark 4.0.0, Scala 2.13)
- **Single Node**
    - Azure: Standard_DS4_v2 (28 GB Memory, 8 Cores)
    - AWS: m5d.2xlarge (32 GB Memory, 8 Cores)
- **Photon Acceleration:** Disabled (Photon boosts Apache Spark workloads; not all ML workloads will see an improvement)

In [0]:
%pip install -r ./requirements.txt --quiet
dbutils.library.restartPython()

In [0]:
import json
import random
import pandas as pd
import matplotlib.pyplot as plt
import scripts.disruption_scenarios as ds_lib

In [0]:
catalog = "supply_chain_stress_test"  # Change here
schema = "data"                       # Change here
volume = "operational"                # Change here

## Load Datasets

These were generated in `05_realistic_operational_data`.

In [0]:
with open(f"/Volumes/{catalog}/{schema}/{volume}/dataset_realistic_simple.json", "r") as f:
    simple = json.load(f)
with open(f"/Volumes/{catalog}/{schema}/{volume}/dataset_realistic_medium.json", "r") as f:
    medium = json.load(f)

## Simple Network

Unlike `generate_data`'s independently-random parameters, this network's inventory/capacity are sized from actual BOM-propagated demand and a per-node criticality tag (derived from how many alternate suppliers exist for that exact material — see `_criticality_from_supplier_counts` in `scripts/realistic_topologies.py`). We disrupt every tier-2/tier-3 node one at a time, with a wider recovery-time range (10-90 days) than the original notebooks' 1-10 days — this calibration is more resilient to short disruptions, so a wider range is needed to see the network's actual breaking points. We also add one material-wide shortage as a demonstration (the simple network is single-region, so no real historical regional event applies to it — see the medium network below for that).

In [0]:
random.seed(777)  # DO NOT CHANGE! kept for reproducibility across runs
rng = random.Random(1)
scenarios_simple = ds_lib.single_supplier_failure_scenarios(simple, rng, ttr_lo=10, ttr_hi=90)
scenarios_simple.append(
    ds_lib.material_shortage_scenario(
        simple, simple["material_types"][0], ttr=60,
        real_world_basis="Illustrative single-material shortage (no real-world citation at this scale).",
    )
)

lost_profit_simple = pd.concat(
    [ds_lib.run_scenario_ttr(simple, s) for s in scenarios_simple], ignore_index=True
)
lost_profit_simple.sort_values(by="lost_profit", ascending=False).head(10)

With this particular seed, only **`T3_13`** ("Vanguard Precision", a `raw_steel_coil` sub-supplier) shows measurable lost profit — its randomly-assigned 85-day recovery time happens to exceed its own ~72-day time-to-survive threshold (see the TTS cell below), so once its buffer runs dry the network starts losing sales. Every other node's randomly-drawn recovery time stays within the buffer this calibration gives it, so they show zero impact. This is a different story than the original small network in `02_stress_testing`, where *most* random single-node disruptions caused some loss — this network is more resilient by construction (BOM-propagated capacity always includes headroom), so the real risk only shows up for prolonged outages at genuinely thin-buffer nodes.

In [0]:
tts_simple = pd.concat(
    [ds_lib.run_scenario_tts(simple, s) for s in scenarios_simple], ignore_index=True
)
merged_simple = lost_profit_simple.merge(
    tts_simple[["scenario_id", "tts"]], on="scenario_id"
)
merged_simple[merged_simple["tts"] < merged_simple["ttr"]][
    ["scenario_id", "ttr", "tts", "lost_profit"]
]

## Medium Network

~700 nodes, region-tagged. We build the same single-supplier baseline (with a 5-45 day range) plus the curated real-world scenario library — on this dataset that resolves to the **2022 Shanghai COVID-19 lockdown** (China) and the **2011 Thailand floods** (Thailand); the other curated events (Taiwan drought, EUV export halt, HBM shortage) don't apply because this network has no Taiwan/Netherlands nodes or HBM memory suppliers — see the printed skip messages below, which is by design (`named_real_world_scenarios` only returns scenarios whose target actually exists in the given dataset).

In [0]:
rng = random.Random(1)
scenarios_medium = ds_lib.single_supplier_failure_scenarios(medium, rng, ttr_lo=5, ttr_hi=45)
named_medium = ds_lib.named_real_world_scenarios(medium)
scenarios_medium += named_medium

lost_profit_medium = pd.concat(
    [ds_lib.run_scenario_ttr(medium, s) for s in scenarios_medium], ignore_index=True
)
lost_profit_medium[lost_profit_medium["scenario_type"] == "single_node"].sort_values(
    by="lost_profit", ascending=False
).head(10)

### Highest-Risk Single Suppliers

Only a small minority of the ~700 nodes show any measurable single-node risk (most randomly-drawn recovery times stay within this network's buffer) — but the ones that do are worth a closer look.

In [0]:
highest_risk = lost_profit_medium[lost_profit_medium["scenario_type"] == "single_node"].sort_values(
    by="lost_profit", ascending=False
).head(5)
for _, row in highest_risk.iterrows():
    node = row["disrupted"][0]
    print(
        f"{node}: region={medium['region'][node]}, material={medium['supplier_material_type'][node]}, "
        f"criticality={medium['criticality'][node]}, ttr={row['ttr']}, lost_profit={row['lost_profit']:.0f}"
    )

### Single-Node vs. Regional vs. Material-Wide

Now let's compare the worst single-node failure against the two regional scenarios. The most striking result: the **2011 Thailand Floods** scenario disrupts *fewer* nodes than the Shanghai lockdown (109 vs. 197) but is nearly **4x costlier than the worst single-supplier failure** — while the Shanghai lockdown, despite disrupting nearly 200 nodes, shows *zero* measurable impact at its 30-day duration. Node count alone doesn't predict risk: what matters is how structurally dependent the network is on that specific region's suppliers, combined with how long the disruption lasts relative to those suppliers' buffers (see the modeling caveat at the top of this notebook).

In [0]:
comparison = lost_profit_medium.groupby("scenario_type")["lost_profit"].max().sort_values(ascending=False)
ax = comparison.plot(kind="bar", figsize=(7, 4), title="Worst-case lost profit by scenario type (medium network)")
ax.set_ylabel("Lost profit")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Time-to-Recover vs. Time-to-Survive

As in `02_stress_testing`, let's find the single-supplier scenarios where the (randomly-assigned) time-to-recover exceeds the network's time-to-survive — these are the nodes actually worth discussing with suppliers about faster recovery, or worth building extra inventory buffer against.

In [0]:
single_only = [s for s in scenarios_medium if s.scenario_type == "single_node"]
top_risk_scenarios = [
    s for s in single_only
    if s.scenario_id in set(highest_risk["scenario_id"])
]
tts_medium = pd.concat(
    [ds_lib.run_scenario_tts(medium, s) for s in top_risk_scenarios], ignore_index=True
)
highest_risk.merge(tts_medium[["scenario_id", "tts"]], on="scenario_id")[
    ["scenario_id", "ttr", "tts", "lost_profit"]
]

## Wrap Up

We compared single-supplier, regional, and material-wide disruption scenarios on two realistic (but still small-to-medium scale) networks. The key takeaway: a network can look resilient against random single-node failures while still being highly exposed to a correlated regional shock — the two require different mitigation strategies (supplier-specific inventory/TTR negotiation vs. geographic sourcing diversification). The next notebook, `07_realistic_stress_testing (complex network)`, runs the same scenario library at Nvidia-like/Apple-like scale (~2,300 nodes each) using Ray, mirroring `03_stress_testing (large network)`.

&copy; 2025 Databricks, Inc. All rights reserved. The source in this notebook is provided subject to the Databricks License [https://databricks.com/db-license-source].  All included or referenced third party libraries are subject to the licenses set forth below.

| library                                | description             | license    | source                                              |
|----------------------------------------|-------------------------|------------|-----------------------------------------------------|
| pyomo | An object-oriented algebraic modeling language in Python for structured optimization problems | BSD-3 | https://pypi.org/project/pyomo/
| highspy | Linear optimization solver (HiGHS) | MIT | https://pypi.org/project/highspy/
| ray | Framework for scaling AI/Python applications | Apache 2.0 | https://github.com/ray-project/ray